# 04 — Modelado (Modeling)

**Fase CRISP-DM:** Modelado.

**Lee de:** `data/processed/train.csv`.

**Produce:** `models/logistic_regression.joblib` + `models/logistic_regression_metadata.json`.

Toda la lógica de entrenamiento vive en `src/models/train_model.py` (pipeline `preprocesador -> SMOTE -> estimador`, usando `imblearn.Pipeline` para que el balanceo nunca se aplique al validar o predecir) y se orquesta desde `src/pipeline.py::train`. Este notebook solo llama a esa función y documenta el resultado.

**Modelo de esta iteración:** regresión logística (baseline). El notebook original importaba también `DecisionTreeClassifier`, `RandomForestClassifier` y `XGBClassifier` sin llegar a entrenarlos; quedan disponibles para una siguiente iteración registrándolos en `src/models/train_model.py::MODEL_REGISTRY` + `config.yaml -> models`, sin tocar `SklearnModelTrainer` (Open/Closed Principle).

**Balanceo:** `SMOTE` sobre el conjunto de entrenamiento, dentro del pipeline (se eligió SMOTE porque, a diferencia de un undersampling, no descarta observaciones de un dataset ya pequeño para robusta).

**Validación:** `StratifiedKFold` (5 folds), semilla `config.yaml -> project.random_seed`.

In [1]:
import sys
sys.path.insert(0, "..")  # para poder importar src/ desde notebooks/

from src.config import load_config
from src import pipeline

config = load_config()
train_result = pipeline.train(config, model_name="logistic_regression", cv_folds=5)

print("Hiperparámetros:", train_result["metadata"]["hyperparams"])
print("Filas de entrenamiento:", train_result["metadata"]["n_train_rows"])
print("\nMétricas promedio de validación cruzada (5 folds):")
for metric, value in train_result["metadata"]["cv_metrics"].items():
    print(f"  {metric}: {value:.3f}")

Hiperparámetros: {'max_iter': 1000, 'C': 1.0, 'class_weight': None}
Filas de entrenamiento: 1070

Métricas promedio de validación cruzada (5 folds):
  accuracy: 0.687
  precision: 0.916
  recall: 0.700
  f1: 0.794
  roc_auc: 0.742


**Hallazgo:** en validación cruzada, la precisión (precision) es alta (~0.92) pero el recall es más moderado (~0.70) — el modelo es conservador clasificando lotes como "especialidad": cuando lo hace, casi siempre acierta, pero deja pasar (falsos negativos) una parte de los lotes que sí lo son. Esto es razonable dado que las variables predictoras son solo de contexto (sin los subpuntajes de catación); el modelo final se evalúa formalmente sobre el conjunto de test en `05_evaluacion.ipynb`.

El modelo entrenado (pipeline completo: preprocesador + SMOTE + regresión logística) y sus metadatos ya quedaron guardados en `models/logistic_regression.joblib` / `models/logistic_regression_metadata.json`.

## Próximos pasos (fuera de esta iteración)

- [ ] Registrar `DecisionTreeClassifier`, `RandomForestClassifier`, `XGBClassifier` en `MODEL_REGISTRY` y comparar contra este baseline.
- [ ] Búsqueda de hiperparámetros (`GridSearchCV`/`RandomizedSearchCV`) para la regresión logística y los modelos adicionales.
- [ ] Tabla comparativa multi-modelo en `reports/tables/` (hoy solo hay un modelo, así que la comparación se limita a CV vs. test).